# Signal Protocol with Module-LWE (M-LWE)

This notebook implements Module-LWE (M-LWE) for use in the continuous asymmetric ratcheting phase of the Signal Protocol. M-LWE provides a balance between the efficiency of Ring-LWE and the security of standard LWE, making it an ideal candidate for post-quantum forward secrecy.

In [1]:
import numpy as np
import time
import hashlib
import os

## 1. Module-LWE Implementation

We implement the core primitives for M-LWE, including polynomial multiplication and the key exchange mechanism.

In [2]:
class ModuleLWE:
    def __init__(self, n=256, k=3, q=7681, eta=2):
        self.n = n
        self.k = k
        self.q = q
        self.eta = eta
        # Generate a public matrix A in the module R_q^k
        self.A = np.random.randint(0, q, (k, k, n))
        
    def cbd(self, size):
        """Centered Binomial Distribution for error/secret sampling."""
        x = np.random.randint(0, 2, (self.eta, *size))
        y = np.random.randint(0, 2, (self.eta, *size))
        return np.sum(x - y, axis=0)

    def poly_mul(self, poly1, poly2):
        """Polynomial multiplication in R_q = Z_q[x]/(x^n + 1)."""
        # Simplified multiplication for demo purposes (using numpy convolution and wrapping)
        res = np.convolve(poly1, poly2)
        n = self.n
        # Reduction modulo x^n + 1
        final_res = (res[:n] - np.concatenate([res[n:], [0]*(n - (len(res)-n))])) % self.q
        return final_res

    def generate_keypair(self):
        s = self.cbd((self.k, self.n))
        e = self.cbd((self.k, self.n))
        
        # t = A * s + e
        t = np.zeros((self.k, self.n), dtype=int)
        for i in range(self.k):
            for j in range(self.k):
                t[i] = (t[i] + self.poly_mul(self.A[i, j], s[j])) % self.q
            t[i] = (t[i] + e[i]) % self.q
        return s, t

    def encapsulate(self, t_pub):
        r = self.cbd((self.k, self.n))
        e1 = self.cbd((self.k, self.n))
        e2 = self.cbd((self.n,))
        
        # u = A^T * r + e1
        u = np.zeros((self.k, self.n), dtype=int)
        for i in range(self.k):
            for j in range(self.k):
                u[i] = (u[i] + self.poly_mul(self.A[j, i], r[j])) % self.q
            u[i] = (u[i] + e1[i]) % self.q
            
        # v = t_pub^T * r + e2 + round(q/2) * msg
        # For KEM, we extract a secret from the high bits of t_pub^T * r
        v = np.zeros((self.n,), dtype=int)
        for j in range(self.k):
            v = (v + self.poly_mul(t_pub[j], r[j])) % self.q
        v = (v + e2) % self.q
        
        return (u, v)

    def decapsulate(self, s_priv, u, v):
        # secret = v - s_priv^T * u
        temp = np.zeros((self.n,), dtype=int)
        for j in range(self.k):
            temp = (temp + self.poly_mul(s_priv[j], u[j])) % self.q
        
        res = (v - temp) % self.q
        # Recover bits (threshold at q/2)
        bits = (res > (self.q // 4)) & (res < (3 * self.q // 4))
        return bits.astype(int)

## 2. Ratchet Step Simulation

Simulation of a DH-like ratchet step using M-LWE.

In [3]:
mlwe = ModuleLWE()

print("1. Bob (Receiver) generates a new M-LWE keypair for the next ratchet step.")
s_bob, t_bob = mlwe.generate_keypair()

print("2. Alice (Sender) encapsulates a new secret using Bob's public key.")
u, v = mlwe.encapsulate(t_bob)

print("3. Bob decapsulates the secret to recover the bits.")
recovered_bits = mlwe.decapsulate(s_bob, u, v)

# Since we didn't add an explicit message, the recovered bits represent the raw shared secret entropy
shared_secret = hashlib.sha256(recovered_bits.tobytes()).digest()
print(f"Established Ratchet Secret: {shared_secret.hex()}")

1. Bob (Receiver) generates a new M-LWE keypair for the next ratchet step.
2. Alice (Sender) encapsulates a new secret using Bob's public key.
3. Bob decapsulates the secret to recover the bits.
Established Ratchet Secret: e5a00aa9991ac8a5ee3109844d84a55583bd20572ad3ffcd42792f3c36b183ad


## 3. Metric Analysis

In [4]:
def analyze_metrics():
    iterations = 50
    total_gen = 0
    total_enc = 0
    total_dec = 0
    
    for _ in range(iterations):
        # Key Gen
        start = time.time()
        s, t = mlwe.generate_keypair()
        total_gen += (time.time() - start)
        
        # Encapsulation
        start = time.time()
        u, v = mlwe.encapsulate(t)
        total_enc += (time.time() - start)
        
        # Decapsulation
        start = time.time()
        bits = mlwe.decapsulate(s, u, v)
        total_dec += (time.time() - start)
        
    print(f"--- Average Execution Times ({iterations} iterations) ---")
    print(f"Key Generation:  {total_gen / iterations * 1000:.3f} ms")
    print(f"Encapsulation:   {total_enc / iterations * 1000:.3f} ms")
    print(f"Decapsulation:   {total_dec / iterations * 1000:.3f} ms")
    
    print("\n--- Communication Overhead (Key/Ciphertext Sizes) ---")
    print(f"Public Key (t):  {t.nbytes} bytes")
    print(f"Ciphertext (u):  {u.nbytes} bytes")
    print(f"Ciphertext (v):  {v.nbytes} bytes")
    print(f"Total Transmitted per Ratchet: {t.nbytes + u.nbytes + v.nbytes} bytes")

analyze_metrics()

--- Average Execution Times (50 iterations) ---
Key Generation:  0.449 ms
Encapsulation:   0.567 ms
Decapsulation:   0.128 ms

--- Communication Overhead (Key/Ciphertext Sizes) ---
Public Key (t):  6144 bytes
Ciphertext (u):  6144 bytes
Ciphertext (v):  2048 bytes
Total Transmitted per Ratchet: 14336 bytes


## 4. Security Analysis

### Hardness Assumption
M-LWE relies on the hardness of finding a secret in a module over a polynomial ring. It is the basis for the CRYSTALS-Kyber (now ML-KEM) standard, selected by NIST for its efficiency and security against quantum computers.

### Application to Double Ratchet
By using M-LWE for the DH-ratchet step, we ensure that if a current state is compromised, the attacker cannot derive future keys even with a quantum computer, provided at least one new M-LWE ratchet step occurs after the compromise. This maintains the **Post-Quantum Future Secrecy** of the protocol.